In [ ]:
import math
import time
import random
from collections import Counter, defaultdict, namedtuple

import numpy as np
import networkx as nx
import pyzx as zx

from GraphGeneration.three_ary_graph import random_3ary_zx_graph
from GraphGeneration.random_graph import random_zx_graph
from GraphGeneration.zx_to_nx import nx_graph, collapse_io, h_boxes_to_edges
from Algorithms.sa import init_order, full_order
from Utils.print_OLA_graphs import plot

# everything that used to be defined in the cells below
from FragmentScheduling import (
    # fragment model and scoring
    FRAGMENTS, FOOTPRINT, NotExtractable, Occupancy,
    check_fragment_table, occupancy_profile, explain_column,
    # graph / schedule primitives
    build_adjacency, timesteps, spider_io, first_high_fan, no_one_sided,
    spider_colour, same_colour, can_place, total_energy,
    # order space
    make_occ_cache, propose_order_swap, propose_order_move, commit_order,
    validate_occ_cache, simulated_annealing_feasibility,
    # layering, refinement, unfusion
    greedy_layers, leg_directions_ok, components, joint_groups_in_layer,
    try_move, refine, move_earliest, co_measure_cleanup,
    unfuse, optimize_depth_with_unfusion,
    # scheduling
    Sched, as_sched, schedule_min_depth,
    sched_serial, sched_asap, sched_capped, sched_cap_sweep, sched_anneal,
    # the ramp and its reports
    progressive_depth_search, compare_ramps,
    ramp_table, ramp_comparison_table, plot_ramp, plot_ramps, show_best,
    # extraction
    extract_schedule, schedule_to_stim, StimExtractor
)

# run-level knobs; rebind any of them in the next-but-one cell
from FragmentScheduling.config import (T_INIT, T_MIN, ALPHA, QUBIT_LIMIT, UNMERGE_AT_BOUNDARY, 
ALLOW_CO_MEASURE, POLISH_VOLUME, SEEDS, SA_KWARGS)

In [ ]:
check_fragment_table()

## Load a diagram

Pick exactly one loader. The 3-ary generator is uncommented so the notebook
runs top to bottom; the original had all three commented out, which left `g`
undefined.

In [ ]:
# LOAD RANDOM

MIN_N_SPIDER, MAX_N_SPIDER = 100, 200
random.seed(23)
IO = random.randint(5, 20)
n_spiders = 434

g, _ = random_3ary_zx_graph(IO, IO, n_spiders = n_spiders, seed = 10, circuit_like = True, layout_jitter = 0.10)
# g, _  = random_zx_graph(n_in = IO, n_out = IO, n_spiders = n_spiders, edge_p = 0.4, seed = None)

# LOAD PRE=MADE
# with open("example_cat_n25_k2_d5.json", "r") as f:
#     g = zx.Graph.from_json(f.read())
#     g, skipped = h_boxes_to_edges(g)
# num_spiders = sum(1 for v in g.vertices() if g.type(v) in (zx.VertexType.Z, zx.VertexType.X))
# print(num_spiders)

zx.draw(g,labels = True)

nxg, pos = nx_graph(g)
nxg_collapsed, pos_collapsed, _, _, colour_map = collapse_io(nxg)

In [ ]:
_init_order, _ = init_order(nxg_collapsed, list(nxg_collapsed.nodes()))
_init_prof = occupancy_profile(nxg_collapsed, [[v] for v in _init_order], build_adjacency(nxg_collapsed)[1])
print(f"initial ordering needs {_init_prof.peak} qubits "
      f"over {_init_prof.depth} columns")

## Settings

The values come from `FragmentScheduling.config`. Rebind them here to
deviate for this run only.

In [ ]:
# T_INIT, T_MIN, ALPHA = 10, 1e-6, 0.995
# QUBIT_LIMIT = 1000
# UNMERGE_AT_BOUNDARY = False   # unfuse one-sided high-fan spiders
# ALLOW_CO_MEASURE = False
# POLISH_VOLUME = True          # second pass: minimise volume at fixed depth
# SEEDS = (23,)                 # e.g. (23, 24, 25, 26) to take the best of four
# SA_KWARGS = dict(T_init = T_INIT, T_min = T_MIN, alpha = ALPHA, steps_per_temp = 20, prob_adj = 0.6)

print(f"q< = {QUBIT_LIMIT}  unmerge = {UNMERGE_AT_BOUNDARY}  co_measure = {ALLOW_CO_MEASURE}  "
      f"polish_volume = {POLISH_VOLUME}  seeds = {SEEDS}")
print(SA_KWARGS)

## Pipeline scheduler

In [ ]:
# RUN_KWARGS = dict(sa_fn=simulated_annealing_feasibility, q_min=46,
#                   q_max = 46, step = 1, target_depth = None, patience = None,
#                   allow_co_measure = ALLOW_CO_MEASURE, use_unfusion = UNMERGE_AT_BOUNDARY, polish_volume = POLISH_VOLUME,
#                   sa_kwargs = SA_KWARGS, seeds = SEEDS)

RUN_KWARGS = dict(sa_fn = simulated_annealing_feasibility, q_min = QUBIT_LIMIT,
                  q_max = QUBIT_LIMIT, step = 1, scheduler = "pipeline",
                  target_depth = None, patience = None,
                  allow_co_measure = ALLOW_CO_MEASURE,
                  use_unfusion = UNMERGE_AT_BOUNDARY,
                  polish_volume = POLISH_VOLUME,
                  sa_kwargs = SA_KWARGS, seeds = SEEDS)

result = progressive_depth_search(nxg_collapsed, colour_map = colour_map, pos = pos_collapsed, **RUN_KWARGS)

In [ ]:
# RUN_KWARGS = dict(sa_fn = simulated_annealing_feasibility, q_min = None,
#                   q_max = QUBIT_LIMIT, step = 1, scheduler = "pipeline",
#                   target_depth = None, patience = None,
#                   allow_co_measure = ALLOW_CO_MEASURE,
#                   use_unfusion = UNMERGE_AT_BOUNDARY,
#                   polish_volume = POLISH_VOLUME,
#                   sa_kwargs = SA_KWARGS, seeds = SEEDS)

# result = progressive_depth_search(nxg_collapsed, colour_map = colour_map, pos = pos_collapsed, **RUN_KWARGS)

In [ ]:
df = ramp_table(result) 
plot_ramp(result)

In [ ]:
# show_best now takes polish_volume / unmerge_at_boundary as arguments
# instead of reading them as notebook globals, and reads best["peak"]
# (best["max_aug"] never existed).
_ = show_best(result, "pipeline",
              allow_co_measure = ALLOW_CO_MEASURE,
              polish_volume = POLISH_VOLUME,
              unmerge_at_boundary = UNMERGE_AT_BOUNDARY,
              draw_labels = False, colour_in = True)

## SIMULATED ANNEALING DEPTH REDUCTION

In [ ]:
DEPTH_BUDGET = 40000

SA_GRAPH = nxg_collapsed
SA_LIMIT = QUBIT_LIMIT
SA_KW = dict(T_init = 8, T_min = 1e-5, alpha = 0.995, steps_per_temp = 20, prob_adj = 0.6, seed = 23)

# find a feasible SA order
sa_order, sa_energy, sa_peak, *_ = simulated_annealing_feasibility(SA_GRAPH, qubit_limit = SA_LIMIT, **SA_KW)
order = [v for v in sa_order if v not in ("I", "O")]
serial_schedule = [[v] for v in order]

adj, adj_w = build_adjacency(SA_GRAPH)

sa_prof = occupancy_profile(SA_GRAPH, serial_schedule, adj_w, SA_LIMIT)

print(f"{'fits' if sa_prof.feasible else 'DOES NOT FIT'}")
print(f"columns {sa_prof.depth}   ({sa_prof.columns[0]} .. {sa_prof.columns[1]})")
print(f"volume  {sa_prof.volume}")
print( "fragments", dict(Counter(FRAGMENTS[k][0] for k in sa_prof.typings.values())))

# # plot(SA_GRAPH, order = serial_schedule, cutwidths = sa_prof.peak, depth = sa_prof.depth,
# #     title_prefix = (f"SA order | {sa_prof.peak} qubits, {sa_prof.depth} columns, volume {sa_prof.volume}, q = {SA_LIMIT}"),
# #     size = 200, draw_labels = True, colour_in = True)


# # minimize depth for the fixed SA order
# # serial execution is the minimum-qubit realization of this order.
# serial_prof = occupancy_profile(SA_GRAPH, serial_schedule, adj_w)
# print( f"order needs {serial_prof.peak} qubits serially over {serial_prof.depth} columns")

# pipeline_res = schedule_min_depth(SA_GRAPH, order, Q, adj, adj_w)
# pipeline = (pipeline_res.schedule if pipeline_res is not None and pipeline_res.feasible else None)
# cap = sched_cap_sweep(SA_GRAPH, order, adj, adj_w, Q)
# annealed = sched_anneal(SA_GRAPH, order, adj, adj_w, Q, budget = DEPTH_BUDGET, seed = 0, start = serial_schedule)
# schedules = {"serial": serial_schedule,  "cap sweep": cap, "notebook pipeline": pipeline, "annealed": annealed}

# print(f"\n  {'schedule':<20}{'columns':>9}{'qubits':>8}{'volume':>8}")

# best = best_prof = None

# for name, schedule in schedules.items():
#     if schedule is None:
#         print(f"  {name:<20}{'infeasible':>9}")
#         continue
#     prof = occupancy_profile(SA_GRAPH, schedule, adj_w, Q)
#     print(f"{name:<20}{prof.depth:>9}{prof.peak:>8}{prof.volume:>8}" + ("" if prof.feasible else " (over budget)"))
#     if prof.feasible and (best_prof is None  or (prof.depth, prof.volume) < (best_prof.depth, best_prof.volume)):
#         best, best_prof = schedule, prof

# if best_prof is None:
#     print("\nnothing feasible at this budget")
# else:
#     print(f"\nbest: {best_prof.depth} columns, {best_prof.peak} qubits, volume {best_prof.volume}")
#     print("fragments", dict(Counter(FRAGMENTS[k][0] for k in best_prof.typings.values())))
#     print("occupancy", best_prof.occ)

#     plot(SA_GRAPH, order = best, cutwidths=best_prof.peak, depth = best_prof.depth, 
#          title_prefix=(f"depth-annealed | {best_prof.peak} qubits, {best_prof.depth} columns, volume {best_prof.volume}, q={Q}"), 
#          size = 200, draw_labels = True, colour_in = True)

In [ ]:
RUN_KWARGS = dict(sa_fn = simulated_annealing_feasibility, q_min = None, q_max = QUBIT_LIMIT, step = 1,
                  scheduler = "anneal", anneal_budget = 60000, anneal_seed = 0, target_depth = None, patience = None,
                  allow_co_measure = ALLOW_CO_MEASURE, use_unfusion = UNMERGE_AT_BOUNDARY, polish_volume = POLISH_VOLUME,
                  sa_kwargs = SA_KWARGS, seeds = SEEDS)

result = progressive_depth_search(nxg_collapsed, colour_map = colour_map,  pos = pos_collapsed, **RUN_KWARGS)
df = ramp_table(result)
plot_ramp(result)

## Extract the best schedule to a stim circuit

`G_best` / `schedule_best` were never bound in the original notebook, and the
extraction result was assigned to `result`, overwriting the ramp result that
the later cells read. Both are fixed here.

In [ ]:
best = result["best"]
G_best, schedule_best = best["graph"], best["schedule"]

circuit, extraction_result = schedule_to_stim(G_best, schedule_best, reuse = "deferred")

print(f"{circuit.num_qubits} qubits, {circuit.num_measurements} measurements")
print(f"occupancy_profile predicted a peak of {best['peak']} over {best['depth']} columns")

In [ ]:
circuit

## Cap sweep scheduler

In [ ]:
# result = progressive_depth_search(
#         nxg_collapsed, simulated_annealing_feasibility,
#         q_min = None, q_max = QUBIT_LIMIT, step = 1, scheduler = "cap sweep",
#         order = None, colour_map = colour_map, pos = pos_collapsed,
#         allow_co_measure = ALLOW_CO_MEASURE,
#         use_unfusion = UNMERGE_AT_BOUNDARY,
#         polish_volume = POLISH_VOLUME, verbose = False)

RUN_KWARGS = dict(sa_fn = simulated_annealing_feasibility, q_min = None,
                  q_max = QUBIT_LIMIT, step = 1, scheduler = "cap sweep",
                  target_depth = None, patience = None,
                  allow_co_measure = ALLOW_CO_MEASURE,
                  use_unfusion = UNMERGE_AT_BOUNDARY,
                  polish_volume = POLISH_VOLUME,
                  sa_kwargs = SA_KWARGS, seeds = SEEDS)

result = progressive_depth_search(nxg_collapsed, colour_map = colour_map, pos = pos_collapsed, **RUN_KWARGS)

In [ ]:
df = ramp_table(result) 
result

In [ ]:
SA_GRAPH = nxg_collapsed
SA_KW = dict(T_init = 10, T_min = 1e-6, alpha = 0.999, steps_per_temp = 20,
             prob_adj = 0.6, seed = 23)

sa_order, sa_energy, sa_peak, _h, _b = simulated_annealing_feasibility(
    SA_GRAPH, qubit_limit = QUBIT_LIMIT, **SA_KW)

order = [v for v in sa_order if v not in ("I", "O")]

adj, adj_w = build_adjacency(SA_GRAPH)
prof = occupancy_profile(SA_GRAPH, [[v] for v in order], adj_w, QUBIT_LIMIT)

print(f"qubits {prof.peak} (SA reported {sa_peak}), columns {prof.depth}, "
      f"volume {prof.volume}, {'fits' if prof.feasible else 'DOES NOT FIT'}")
print("fragments", dict(Counter(FRAGMENTS[k][0] for k in prof.typings.values())))

In [ ]:
best = result["best"]

print(f"cap sweep: {best['depth']} columns, {best['peak']} qubits, volume {best['volume']}, at q={best['q']}")

plot(nxg_collapsed, order = best["schedule"], cutwidths = best["peak"], depth = best["depth"],
     title_prefix = (f"cap sweep | {best['peak']} qubits, {best['depth']} columns, volume {best['volume']}, q = {best['q']}"), 
     size = 10, draw_labels = False, colour_in = True)